In [23]:
import os

In [24]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [25]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

In [26]:
from langchain_huggingface import HuggingFaceEmbeddings

In [27]:
from langchain_community.vectorstores.faiss import FAISS

In [28]:
KNOWLEDGE_BASE_DIR = "knowledge_base"

In [29]:
FAISS_INDEX_DIR = "faiss_index_multi"

In [30]:
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

In [31]:
print("--- Начало процесса создания индекса FAISS ---")

--- Начало процесса создания индекса FAISS ---


In [32]:
loader_kwargs = {'encoding': 'utf-8'}

In [33]:
loader = DirectoryLoader(
    KNOWLEDGE_BASE_DIR,
    glob="**/*.txt",
    loader_cls=TextLoader,
    show_progress=True,
    use_multithreading=True,
    loader_kwargs=loader_kwargs
)

In [34]:
documents = loader.load()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 36/36 [00:00<00:00, 3778.84it/s]


In [35]:
if not documents:
    print(f"Ошибка: Не найдены документы в директории {KNOWLEDGE_BASE_DIR}")
    exit()

In [36]:
print(f"Загружено {len(documents)} документов.")

Загружено 36 документов.


In [37]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [38]:
chunks = text_splitter.split_documents(documents)
print(f"Документы разбиты на {len(chunks)} чанков.")

Документы разбиты на 237 чанков.


In [39]:
print(f"Загрузка эмбеддинг-модели: {EMBEDDING_MODEL_NAME}...")

Загрузка эмбеддинг-модели: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...


In [40]:
print(f"Загрузка эмбеддинг-модели: {EMBEDDING_MODEL_NAME}...")
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print("Модель успешно загружена.")

Загрузка эмбеддинг-модели: sentence-transformers/paraphrase-multilingual-mpnet-base-v2...
Модель успешно загружена.


In [42]:
print("Создание FAISS индекса в памяти... (это может занять время)")
db = FAISS.from_documents(chunks, embedding_model)

Создание FAISS индекса в памяти... (это может занять время)


In [43]:
print(f"Сохранение индекса на диск в директорию: {FAISS_INDEX_DIR}...")
# Сохраняем индекс и карту документов локально
db.save_local(FAISS_INDEX_DIR)

Сохранение индекса на диск в директорию: faiss_index_multi...


In [44]:
print("--- Процесс создания индекса FAISS успешно завершен! ---")
print(f"Индекс сохранен в папке: {FAISS_INDEX_DIR}")

--- Процесс создания индекса FAISS успешно завершен! ---
Индекс сохранен в папке: faiss_index_multi
